<a href="https://colab.research.google.com/github/PoojaBodekar/CSL701-CS07-Machine-Learning-Lab/blob/main/CS07_ML_Experiment_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Student Name : Pooja Suresh Bodekar

Batch : CS01


Roll Number : CS07



### **Aim**

To implement graph-based clustering using Minimum Spanning Tree (MST) on the Breast Cancer Wisconsin (Diagnostic) dataset in Python and visualize the resulting clusters.

## Objectives

1. To understand the concept of graph-based clustering and Minimum Spanning Trees (MST).
2. To load and preprocess the Breast Cancer Wisconsin dataset (scaling features and handling metadata).
3. To construct a distance graph from tumor feature representations using Euclidean distance.
4. To compute the Minimum Spanning Tree using Prim's or Kruskal's algorithm (or via scipy/networkx).
5. To form $K=2$ clusters (Malignant vs. Benign) by removing the single ($K - 1$) largest edge weight from the MST.
6. To evaluate the performance of the MST-based clustering model using evaluation metrics.

# Relevant Theory

###Graph-Based Clustering & Minimum Spanning Tree (MST)
Graph-based clustering represents data points as nodes in a graph, with edges representing relationships or distance metrics between individual samples.
* **Complete Graph Construction**: Given a dataset $X = \{x_1, x_2, \dots, x_n\}$,
each tissue sample is treated as a vertex. An edge weight $w(i, j)$ corresponds to the distance between sample $i$ and sample $j$ (typically Euclidean distance on scaled features):$$w(i, j) = \Vert{}x_i - x_j\Vert{}_2$$
* **Minimum Spanning Tree (MST)**: An MST is a subset of edges connecting all vertices in the graph without forming any cycles, while minimizing the total edge weight sum.
* **MST-Based Clustering Method**:
  1. Construct a fully connected graph with pairwise edge weights across standardized features.
  2. Compute the Minimum Spanning Tree (MST).
  3. To partition the data into $K$ distinct clusters (such as Malignant and Benign groups), identify and remove the $K - 1$ largest (most expensive) edges from the MST.4. The resulting disconnected subgraphs represent the separate data clusters.
### Dataset Information
Get the Breast Cancer Wisconsin (Diagnostic) dataset from Kaggle or via scikit-learn (load_breast_cancer).
The dataset contains key cell nuclei features computed from digitized images of fine needle aspirates (FNA) of breast masses:
  * Total Instances: 569 samples
  * Target Classes: Malignant (M) and Benign (B)* Feature Attributes: 30 numerical features (e.g., mean radius, mean texture, mean perimeter, mean area, mean smoothness)

For 2D visualization, two key features (such as mean radius and mean texture) will be selected.

# Task 1: Import Libraries and Load Dataset

* Import required libraries (numpy, pandas, matplotlib, scipy, networkx, sklearn).

* Load the Breast Cancer dataset using below link from Kaggle.

  https://www.kaggle.com/datasets/utkarshx27/breast-cancer-wisconsin-diagnostic-dataset

---

In [ ]:
# ============================================
# Task 1: Import Libraries and Load Dataset
# ============================================

# Core Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Graph and MST libraries
import networkx as nx
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ============================================
# Load Dataset
# ============================================

# If df already exists from the previous experiment,
# use it directly.
try:
    df
    print("Dataset already loaded.")

except NameError:
    # If df does not exist, upload the CSV file
    from google.colab import files

    uploaded = files.upload()

    file_name = list(uploaded.keys())[0]

    df = pd.read_csv(file_name)

# ============================================
# Display Dataset
# ============================================

print("\nFirst 5 rows of the dataset:")
display(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()


Saving brca.csv to brca.csv

First 5 rows of the dataset:


,Unnamed: 0,x.radius_mean,x.texture_mean,x.perimeter_mean,x.area_mean,x.smoothness_mean,x.compactness_mean,x.concavity_mean,x.concave_pts_mean,x.symmetry_mean,...,x.texture_worst,x.perimeter_worst,x.area_worst,x.smoothness_worst,x.compactness_worst,x.concavity_worst,x.concave_pts_worst,x.symmetry_worst,x.fractal_dim_worst,y
0,1,13.540,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.047810,0.1885,...,19.26,99.70,711.2,0.14400,0.17730,0.23900,0.12880,0.2977,0.07259,B
1,2,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.031100,0.1967,...,20.49,96.09,630.5,0.13120,0.27760,0.18900,0.07283,0.3184,0.08183,B
2,3,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.020760,0.1815,...,15.66,65.13,314.9,0.13240,0.11480,0.08867,0.06227,0.2450,0.07773,B
3,4,13.030,18.42,82.61,523.8,0.08983,0.03766,0.02562,0.029230,0.1467,...,22.81,84.46,545.9,0.09701,0.04619,0.04833,0.05013,0.1987,0.06169,B
4,5,8.196,16.84,51.71,201.9,0.08600,0.05943,0.01588,0.005917,0.1769,...,21.96,57.26,242.2,0.12970,0.13570,0.06880,0.02564,0.3105,0.07409,B



Dataset shape:
(569, 32)

Column names:
['Unnamed: 0', 'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.area_mean', 'x.smoothness_mean', 'x.compactness_mean', 'x.concavity_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.radius_se', 'x.texture_se', 'x.perimeter_se', 'x.area_se', 'x.smoothness_se', 'x.compactness_se', 'x.concavity_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.texture_worst', 'x.perimeter_worst', 'x.area_worst', 'x.smoothness_worst', 'x.compactness_worst', 'x.concavity_worst', 'x.concave_pts_worst', 'x.symmetry_worst', 'x.fractal_dim_worst', 'y']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           569 non-null    int64  
 1   x.radius_mean        569 non-null    float64
 2   x.texture_mean       569 non-

# Task 2: Explore the Dataset

* Display missing value checks, summary statistics, and feature dimensions.

* Check the diagnosis class distribution (Malignant vs. Benign).

In [ ]:
# ============================================
# Task 2: Explore the Dataset
# ============================================

# Clean column names
df_mst.columns = df_mst.columns.str.strip()

# --------------------------------------------
# 1. Dataset dimensions
# --------------------------------------------

print("===== Dataset Dimensions =====")
print("Number of samples :", df_mst.shape[0])
print("Number of columns :", df_mst.shape[1])

# --------------------------------------------
# 2. Display first 5 rows
# --------------------------------------------

print("\n===== First 5 Rows =====")
display(df_mst.head())

# --------------------------------------------
# 3. Check missing values
# --------------------------------------------

print("\n===== Missing Value Check =====")

missing_values = df_mst.isnull().sum()

display(missing_values[missing_values > 0])

print("Total missing values:", df_mst.isnull().sum().sum())

# --------------------------------------------
# 4. Summary statistics
# --------------------------------------------

print("\n===== Summary Statistics =====")

display(df_mst.describe())

# --------------------------------------------
# 5. Data types
# --------------------------------------------

print("\n===== Data Types =====")

print(df_mst.dtypes)

# --------------------------------------------
# 6. Find diagnosis column safely
# --------------------------------------------

diagnosis_col = None

for col in df_mst.columns:
    if col.lower().strip() == "diagnosis":
        diagnosis_col = col
        break

if diagnosis_col is None:

    print("\nERROR: Diagnosis column was not found.")

    print("Available columns:")
    print(df_mst.columns.tolist())

else:

    print("\nDiagnosis column found:", diagnosis_col)

    # ----------------------------------------
    # 7. Diagnosis class distribution
    # ----------------------------------------

    print("\n===== Diagnosis Class Distribution =====")

    diagnosis_counts = df_mst[diagnosis_col].value_counts()

    print(diagnosis_counts)

    # ----------------------------------------
    # 8. Diagnosis percentages
    # ----------------------------------------

    print("\n===== Diagnosis Distribution (%) =====")

    diagnosis_percent = (
        df_mst[diagnosis_col]
        .value_counts(normalize=True) * 100
    )

    print(diagnosis_percent.round(2))

    # ----------------------------------------
    # 9. Convert labels to meaningful names
    # ----------------------------------------

    print("\n===== Class Description =====")

    print("B = Benign")
    print("M = Malignant")

    # ----------------------------------------
    # 10. Plot class distribution
    # ----------------------------------------

    plt.figure(figsize=(7, 5))

    sns.countplot(
        data=df_mst,
        x=diagnosis_col,
        hue=diagnosis_col,
        palette={
            "B": "steelblue",
            "M": "tomato"
        },
        legend=False
    )

    plt.title(
        "Breast Cancer Diagnosis Distribution",
        fontsize=14
    )

    plt.xlabel("Diagnosis")
    plt.ylabel("Number of Samples")

    plt.xticks(
        [0, 1],
        ["Benign (B)", "Malignant (M)"]
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.tight_layout()
    plt.show()

# --------------------------------------------
# 11. Identify the 30 numerical features
# --------------------------------------------

metadata_columns = [
    "id",
    "diagnosis",
    "Unnamed: 32"
]

feature_columns = [
    col for col in df_mst.columns
    if col not in metadata_columns
]

# Keep only numerical columns
numerical_features = df_mst[feature_columns].select_dtypes(
    include=np.number
).columns

print("\n===== Feature Dimensions =====")

print("Number of numerical features:",
      len(numerical_features))

print("Number of samples:",
      df_mst.shape[0])

print(
    f"Feature matrix dimensions: "
    f"{df_mst.shape[0]} × {len(numerical_features)}"
)

print("\nNumerical Feature Names:")
print(numerical_features.tolist())


===== Dataset Dimensions =====
Number of samples : 569
Number of columns : 32

===== First 5 Rows =====


,Unnamed: 0,x.radius_mean,x.texture_mean,x.perimeter_mean,x.area_mean,x.smoothness_mean,x.compactness_mean,x.concavity_mean,x.concave_pts_mean,x.symmetry_mean,...,x.texture_worst,x.perimeter_worst,x.area_worst,x.smoothness_worst,x.compactness_worst,x.concavity_worst,x.concave_pts_worst,x.symmetry_worst,x.fractal_dim_worst,y
0,1,13.540,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.047810,0.1885,...,19.26,99.70,711.2,0.14400,0.17730,0.23900,0.12880,0.2977,0.07259,B
1,2,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.031100,0.1967,...,20.49,96.09,630.5,0.13120,0.27760,0.18900,0.07283,0.3184,0.08183,B
2,3,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.020760,0.1815,...,15.66,65.13,314.9,0.13240,0.11480,0.08867,0.06227,0.2450,0.07773,B
3,4,13.030,18.42,82.61,523.8,0.08983,0.03766,0.02562,0.029230,0.1467,...,22.81,84.46,545.9,0.09701,0.04619,0.04833,0.05013,0.1987,0.06169,B
4,5,8.196,16.84,51.71,201.9,0.08600,0.05943,0.01588,0.005917,0.1769,...,21.96,57.26,242.2,0.12970,0.13570,0.06880,0.02564,0.3105,0.07409,B



===== Missing Value Check =====


,0


Total missing values: 0

===== Summary Statistics =====


,Unnamed: 0,x.radius_mean,x.texture_mean,x.perimeter_mean,x.area_mean,x.smoothness_mean,x.compactness_mean,x.concavity_mean,x.concave_pts_mean,x.symmetry_mean,...,x.radius_worst,x.texture_worst,x.perimeter_worst,x.area_worst,x.smoothness_worst,x.compactness_worst,x.concavity_worst,x.concave_pts_worst,x.symmetry_worst,x.fractal_dim_worst
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,285.000000,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,164.400426,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,1.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,143.000000,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,285.000000,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,427.000000,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,569.000000,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500



===== Data Types =====
Unnamed: 0               int64
x.radius_mean          float64
x.texture_mean         float64
x.perimeter_mean       float64
x.area_mean            float64
x.smoothness_mean      float64
x.compactness_mean     float64
x.concavity_mean       float64
x.concave_pts_mean     float64
x.symmetry_mean        float64
x.fractal_dim_mean     float64
x.radius_se            float64
x.texture_se           float64
x.perimeter_se         float64
x.area_se              float64
x.smoothness_se        float64
x.compactness_se       float64
x.concavity_se         float64
x.concave_pts_se       float64
x.symmetry_se          float64
x.fractal_dim_se       float64
x.radius_worst         float64
x.texture_worst        float64
x.perimeter_worst      float64
x.area_worst           float64
x.smoothness_worst     float64
x.compactness_worst    float64
x.concavity_worst      float64
x.concave_pts_worst    float64
x.symmetry_worst       float64
x.fractal_dim_worst    float64
y              

# Task 3: Preprocess and Scale Features

* Select numerical features and drop non-informative identifiers (such as id).

* Apply StandardScaler to normalize feature magnitudes prior to graph distance calculation.

In [ ]:
# ============================================
# Task 3: Preprocess and Scale Features
# ============================================

from sklearn.preprocessing import StandardScaler

# --------------------------------------------
# 1. Remove non-informative columns
# --------------------------------------------

# Drop ID, diagnosis, and empty column
X = df_mst.drop(
    columns=["id", "diagnosis", "Unnamed: 32"],
    errors="ignore"
)

# --------------------------------------------
# 2. Select numerical features
# --------------------------------------------

X = X.select_dtypes(include=[np.number])

# Display feature information
print("Feature matrix shape:")
print(X.shape)

print("\nNumber of features:")
print(X.shape[1])

print("\nFeature names:")
print(X.columns.tolist())

# --------------------------------------------
# 3. Check for missing values
# --------------------------------------------

print("\nTotal missing values in features:")
print(X.isnull().sum().sum())

# --------------------------------------------
# 4. Standardize the features
# --------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for easier inspection
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

# --------------------------------------------
# 5. Display standardized data
# --------------------------------------------

print("\nFirst 5 rows of standardized features:")
display(X_scaled.head())

# --------------------------------------------
# 6. Verify mean and standard deviation
# --------------------------------------------

print("\nMean of standardized features:")
print(X_scaled.mean().round(6))

print("\nStandard deviation of standardized features:")
print(X_scaled.std(ddof=0).round(6))

# --------------------------------------------
# 7. Final dimensions
# --------------------------------------------

print("\nFinal standardized feature matrix shape:")
print(X_scaled.shape)


Feature matrix shape:
(569, 31)

Number of features:
31

Feature names:
['Unnamed: 0', 'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.area_mean', 'x.smoothness_mean', 'x.compactness_mean', 'x.concavity_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.radius_se', 'x.texture_se', 'x.perimeter_se', 'x.area_se', 'x.smoothness_se', 'x.compactness_se', 'x.concavity_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.texture_worst', 'x.perimeter_worst', 'x.area_worst', 'x.smoothness_worst', 'x.compactness_worst', 'x.concavity_worst', 'x.concave_pts_worst', 'x.symmetry_worst', 'x.fractal_dim_worst']

Total missing values in features:
0

First 5 rows of standardized features:


,Unnamed: 0,x.radius_mean,x.texture_mean,x.perimeter_mean,x.area_mean,x.smoothness_mean,x.compactness_mean,x.concavity_mean,x.concave_pts_mean,x.symmetry_mean,...,x.radius_worst,x.texture_worst,x.perimeter_worst,x.area_worst,x.smoothness_worst,x.compactness_worst,x.concavity_worst,x.concave_pts_worst,x.symmetry_worst,x.fractal_dim_worst
0,-1.729009,-0.166799,-1.147162,-0.185728,-0.251957,0.101747,-0.436850,-0.278210,-0.028609,0.267911,...,-0.240048,-1.045005,-0.225217,-0.297761,0.509873,-0.489605,-0.159223,0.216123,0.123347,-0.629292
1,-1.722921,-0.297446,-0.833008,-0.261106,-0.383638,0.792763,0.429422,-0.541362,-0.459627,0.567289,...,-0.366368,-0.844707,-0.332744,-0.439624,-0.051226,0.148443,-0.399099,-0.636110,0.458227,-0.117250
2,-1.716833,-1.313080,-1.593959,-1.302806,-1.083572,0.429819,-0.747086,-0.743748,-0.726337,0.012345,...,-1.250611,-1.631243,-1.254913,-0.994422,0.001377,-0.887193,-0.880434,-0.796903,-0.729224,-0.344455
3,-1.710745,-0.311646,-0.202373,-0.385500,-0.372831,-0.464730,-1.263703,-0.793214,-0.507861,-1.258183,...,-0.614867,-0.466909,-0.679153,-0.588344,-1.549975,-1.323648,-1.073966,-0.981753,-1.478256,-1.233324
4,-1.704657,-1.684571,-0.570050,-1.658278,-1.288347,-0.737294,-0.851130,-0.915500,-1.109197,-0.155598,...,-1.512777,-0.605327,-1.489328,-1.122222,-0.116980,-0.754239,-0.975761,-1.354653,0.330422,-0.546168



Mean of standardized features:
Unnamed: 0             0.0
x.radius_mean         -0.0
x.texture_mean        -0.0
x.perimeter_mean      -0.0
x.area_mean           -0.0
x.smoothness_mean      0.0
x.compactness_mean     0.0
x.concavity_mean       0.0
x.concave_pts_mean     0.0
x.symmetry_mean        0.0
x.fractal_dim_mean     0.0
x.radius_se            0.0
x.texture_se          -0.0
x.perimeter_se         0.0
x.area_se             -0.0
x.smoothness_se       -0.0
x.compactness_se       0.0
x.concavity_se        -0.0
x.concave_pts_se       0.0
x.symmetry_se          0.0
x.fractal_dim_se      -0.0
x.radius_worst        -0.0
x.texture_worst        0.0
x.perimeter_worst      0.0
x.area_worst          -0.0
x.smoothness_worst    -0.0
x.compactness_worst    0.0
x.concavity_worst      0.0
x.concave_pts_worst    0.0
x.symmetry_worst       0.0
x.fractal_dim_worst    0.0
dtype: float64

Standard deviation of standardized features:
Unnamed: 0             1.0
x.radius_mean          1.0
x.texture_mean  

# Task 4: Construct Pairwise Distance Matrix & Graph

* Compute pairwise Euclidean distance matrix across all scaled samples using scipy.spatial.distance.pdist or cdist.

* Represent the dataset as a weighted complete graph.

In [ ]:
# ============================================
# Task 4: Pairwise Distance Matrix & Graph
# ============================================

from scipy.spatial.distance import pdist, squareform
import networkx as nx
import numpy as np

# --------------------------------------------
# 1. Convert scaled data to NumPy array
# --------------------------------------------

X_scaled_array = X_scaled.to_numpy()

print("Scaled feature matrix shape:")
print(X_scaled_array.shape)

# --------------------------------------------
# 2. Compute pairwise Euclidean distances
# --------------------------------------------

# pdist calculates the distance between every
# pair of samples
pairwise_distances = pdist(
    X_scaled_array,
    metric="euclidean"
)

# Convert condensed distance vector into
# a square distance matrix
distance_matrix = squareform(pairwise_distances)

# --------------------------------------------
# 3. Display distance matrix information
# --------------------------------------------

print("\n===== Distance Matrix =====")

print("Distance matrix shape:")
print(distance_matrix.shape)

print("\nFirst 5 × 5 distance matrix:")
print(np.round(distance_matrix[:5, :5], 3))

# --------------------------------------------
# 4. Verify distance matrix
# --------------------------------------------

print("\n===== Distance Matrix Verification =====")

# Distance from a point to itself should be 0
print(
    "Diagonal values are zero:",
    np.allclose(np.diag(distance_matrix), 0)
)

# Distance matrix should be symmetric
print(
    "Matrix is symmetric:",
    np.allclose(
        distance_matrix,
        distance_matrix.T
    )
)

print(
    "Minimum distance:",
    round(distance_matrix.min(), 4)
)

print(
    "Maximum distance:",
    round(distance_matrix.max(), 4)
)

# --------------------------------------------
# 5. Construct weighted complete graph
# --------------------------------------------

G = nx.Graph()

# Add 569 samples as graph nodes
n_samples = X_scaled.shape[0]

G.add_nodes_from(range(n_samples))

# Add an edge between every pair of samples
for i in range(n_samples):

    for j in range(i + 1, n_samples):

        G.add_edge(
            i,
            j,
            weight=distance_matrix[i, j]
        )

# --------------------------------------------
# 6. Display graph information
# --------------------------------------------

print("\n===== Complete Weighted Graph =====")

print("Number of nodes:",
      G.number_of_nodes())

print("Number of edges:",
      G.number_of_edges())

# Number of edges in a complete graph:
# n(n-1)/2

expected_edges = (
    n_samples * (n_samples - 1) // 2
)

print("Expected number of edges:",
      expected_edges)

print(
    "Is graph connected:",
    nx.is_connected(G)
)

# --------------------------------------------
# 7. Display a few graph edges
# --------------------------------------------

print("\n===== First 10 Graph Edges =====")

for i, (node1, node2, data) in enumerate(
    G.edges(data=True)
):

    if i >= 10:
        break

    print(
        f"Node {node1} -- Node {node2} "
        f"| Distance = {data['weight']:.4f}"
    )


Scaled feature matrix shape:
(569, 31)

===== Distance Matrix =====
Distance matrix shape:
(569, 569)

First 5 × 5 distance matrix:
[[0.    3.107 3.623 5.205 4.467]
 [3.107 0.    4.01  5.623 4.371]
 [3.623 4.01  0.    5.122 2.948]
 [5.205 5.623 5.122 0.    5.028]
 [4.467 4.371 2.948 5.028 0.   ]]

===== Distance Matrix Verification =====
Diagonal values are zero: True
Matrix is symmetric: True
Minimum distance: 0.0
Maximum distance: 26.991

===== Complete Weighted Graph =====
Number of nodes: 569
Number of edges: 161596
Expected number of edges: 161596
Is graph connected: True

===== First 10 Graph Edges =====
Node 0 -- Node 1 | Distance = 3.1073
Node 0 -- Node 2 | Distance = 3.6232
Node 0 -- Node 3 | Distance = 5.2045
Node 0 -- Node 4 | Distance = 4.4672
Node 0 -- Node 5 | Distance = 2.5423
Node 0 -- Node 6 | Distance = 3.6177
Node 0 -- Node 7 | Distance = 4.3633
Node 0 -- Node 8 | Distance = 3.9885
Node 0 -- Node 9 | Distance = 3.1456
Node 0 -- Node 10 | Distance = 3.1251


# Task 5: Compute the Minimum Spanning Tree (MST)

* Compute the MST using scipy.sparse.csgraph.minimum_spanning_tree or networkx.minimum_spanning_tree.

* Extract the resulting graph connections and weights.

In [ ]:
# ============================================
# Task 5: Compute Minimum Spanning Tree (MST)
# ============================================

from scipy.sparse.csgraph import minimum_spanning_tree
import networkx as nx
import pandas as pd
import numpy as np

# --------------------------------------------
# 1. Compute MST using SciPy
# --------------------------------------------

mst_sparse = minimum_spanning_tree(distance_matrix)

# Convert sparse matrix to dense matrix
mst_matrix = mst_sparse.toarray()

# --------------------------------------------
# 2. Create NetworkX MST graph
# --------------------------------------------

MST = nx.Graph()

# Add all samples as nodes
n_samples = X_scaled.shape[0]

MST.add_nodes_from(range(n_samples))

# --------------------------------------------
# 3. Extract MST edges and weights
# --------------------------------------------

mst_edges = []

rows, cols = np.nonzero(mst_matrix)

for i, j in zip(rows, cols):

    weight = mst_matrix[i, j]

    # Add edge to NetworkX MST
    MST.add_edge(
        int(i),
        int(j),
        weight=float(weight)
    )

    # Store connection and weight
    mst_edges.append(
        (int(i), int(j), float(weight))
    )

# --------------------------------------------
# 4. Display MST information
# --------------------------------------------

print("==========================================")
print("Minimum Spanning Tree (MST)")
print("==========================================")

print("Number of nodes:",
      MST.number_of_nodes())

print("Number of MST edges:",
      MST.number_of_edges())

print("Expected MST edges:",
      n_samples - 1)

print(
    "Is MST connected:",
    nx.is_connected(MST)
)

print(
    "Is MST a tree:",
    nx.is_tree(MST)
)

# --------------------------------------------
# 5. Extract MST edge table
# --------------------------------------------

mst_edges_df = pd.DataFrame(
    mst_edges,
    columns=[
        "Node_1",
        "Node_2",
        "Weight"
    ]
)

# Sort edges by weight
mst_edges_df = mst_edges_df.sort_values(
    by="Weight",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------
# 6. Display largest MST edges
# --------------------------------------------

print("\n===== 10 Largest MST Edges =====")

display(
    mst_edges_df.head(10)
)

# --------------------------------------------
# 7. Display smallest MST edges
# --------------------------------------------

print("\n===== 10 Smallest MST Edges =====")

display(
    mst_edges_df.tail(10)
)

# --------------------------------------------
# 8. MST total weight
# --------------------------------------------

total_mst_weight = mst_edges_df["Weight"].sum()

print("\nTotal MST Weight:")
print(round(total_mst_weight, 4))


Minimum Spanning Tree (MST)
Number of nodes: 569
Number of MST edges: 568
Expected MST edges: 568
Is MST connected: True
Is MST a tree: True

===== 10 Largest MST Edges =====


,Node_1,Node_2,Weight
0,544,553,12.300067
1,18,69,10.480675
2,468,527,9.042305
3,467,544,8.281820
4,424,429,8.078042
5,360,454,7.891816
6,18,208,7.876494
7,412,515,7.117204
8,454,515,7.072387
9,369,395,6.970796



===== 10 Smallest MST Edges =====


,Node_1,Node_2,Weight
558,204,275,1.390828
559,23,31,1.352185
560,34,49,1.345374
561,74,149,1.300781
562,248,319,1.290774
563,180,188,1.270318
564,21,23,1.230502
565,132,219,1.218262
566,21,57,1.169433
567,271,272,1.027632



Total MST Weight:
1450.2935


# Task 6: Implement Graph-Based Clustering ($K=2$)

* Sort the edges of the MST in descending order by edge weight.
* Remove the largest $K - 1$ edge (1 largest edge for 2 clusters) to split the MST into 2 connected components.
* Assign final cluster labels based on the resulting component indices.


In [ ]:
# ============================================
# Task 6: Graph-Based Clustering (K = 2)
# ============================================

import numpy as np
import networkx as nx

# --------------------------------------------
# 1. Sort MST edges in descending order
# --------------------------------------------

mst_edges_sorted = sorted(
    mst_edges,
    key=lambda edge: edge[2],
    reverse=True
)

print("===== Largest MST Edges =====")

for i, (node1, node2, weight) in enumerate(
    mst_edges_sorted[:10],
    start=1
):
    print(
        f"{i}. Node {node1} -- Node {node2} "
        f"| Weight = {weight:.4f}"
    )

# --------------------------------------------
# 2. Select the largest edge
# --------------------------------------------

largest_edge = mst_edges_sorted[0]

node1 = largest_edge[0]
node2 = largest_edge[1]
largest_weight = largest_edge[2]

print("\n===== Largest MST Edge =====")

print(
    f"Node {node1} -- Node {node2} "
    f"| Weight = {largest_weight:.4f}"
)

# --------------------------------------------
# 3. Copy the original MST
# --------------------------------------------

MST_cluster = MST.copy()

# --------------------------------------------
# 4. Remove the largest edge
# --------------------------------------------

MST_cluster.remove_edge(
    node1,
    node2
)

print("\nLargest edge removed successfully.")

# --------------------------------------------
# 5. Find connected components
# --------------------------------------------

components = list(
    nx.connected_components(MST_cluster)
)

print("\n===== Connected Components =====")

print(
    "Number of connected components:",
    len(components)
)

# --------------------------------------------
# 6. Check that K = 2
# --------------------------------------------

if len(components) != 2:
    raise ValueError(
        f"Expected 2 clusters, but found {len(components)}"
    )

# --------------------------------------------
# 7. Assign cluster labels
# --------------------------------------------

cluster_labels = np.zeros(
    X_scaled.shape[0],
    dtype=int
)

for cluster_id, component in enumerate(components):

    for node in component:

        cluster_labels[node] = cluster_id

# --------------------------------------------
# 8. Display cluster sizes
# --------------------------------------------

print("\n===== Cluster Sizes =====")

for cluster_id in range(2):

    count = np.sum(
        cluster_labels == cluster_id
    )

    print(
        f"Cluster {cluster_id}: {count} samples"
    )

# --------------------------------------------
# 9. Verify all samples received a label
# --------------------------------------------

print("\n===== Cluster Label Verification =====")

print(
    "Total samples:",
    len(cluster_labels)
)

print(
    "Unique cluster labels:",
    np.unique(cluster_labels)
)

print(
    "All samples assigned:",
    len(cluster_labels) == X_scaled.shape[0]
)

# --------------------------------------------
# 10. Create final clustering DataFrame
# --------------------------------------------

cluster_results = pd.DataFrame({
    "Sample_Index": np.arange(
        X_scaled.shape[0]
    ),
    "MST_Cluster": cluster_labels
})

print("\n===== First 10 Cluster Assignments =====")

display(
    cluster_results.head(10)
)

# --------------------------------------------
# 11. Display cluster distribution
# --------------------------------------------

print("\n===== Cluster Distribution =====")

display(
    cluster_results["MST_Cluster"]
    .value_counts()
    .sort_index()
)


===== Largest MST Edges =====
1. Node 544 -- Node 553 | Weight = 12.3001
2. Node 18 -- Node 69 | Weight = 10.4807
3. Node 468 -- Node 527 | Weight = 9.0423
4. Node 467 -- Node 544 | Weight = 8.2818
5. Node 424 -- Node 429 | Weight = 8.0780
6. Node 360 -- Node 454 | Weight = 7.8918
7. Node 18 -- Node 208 | Weight = 7.8765
8. Node 412 -- Node 515 | Weight = 7.1172
9. Node 454 -- Node 515 | Weight = 7.0724
10. Node 369 -- Node 395 | Weight = 6.9708

===== Largest MST Edge =====
Node 544 -- Node 553 | Weight = 12.3001

Largest edge removed successfully.

===== Connected Components =====
Number of connected components: 2

===== Cluster Sizes =====
Cluster 0: 567 samples
Cluster 1: 2 samples

===== Cluster Label Verification =====
Total samples: 569
Unique cluster labels: [0 1]
All samples assigned: True

===== First 10 Cluster Assignments =====


,Sample_Index,MST_Cluster
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0



===== Cluster Distribution =====


,count
MST_Cluster,
0,567
1,2


# Task 7: valuate MST Clustering Performance

Evaluate cluster assignment against ground truth labels using:

  *  Silhouette Score

  *  Adjusted Rand Index (ARI)

  *  Normalized Mutual Information (NMI)

In [ ]:
# ============================================
# Task 7: Evaluate MST Clustering Performance
# ============================================

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# --------------------------------------------
# 1. Find diagnosis column safely
# --------------------------------------------

df_mst.columns = df_mst.columns.str.strip()

diagnosis_col = None

for col in df_mst.columns:
    if col.lower().strip() == "diagnosis":
        diagnosis_col = col
        break

if diagnosis_col is None:
    print("ERROR: Diagnosis column not found.")
    print("\nAvailable columns:")
    print(df_mst.columns.tolist())

else:

    print("Diagnosis column found:", diagnosis_col)

    # ----------------------------------------
    # 2. Convert diagnosis to numerical labels
    # ----------------------------------------

    # B = Benign = 0
    # M = Malignant = 1

    true_labels = (
        df_mst[diagnosis_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .map({
            "B": 0,
            "M": 1
        })
        .to_numpy()
    )

    # ----------------------------------------
    # 3. Check labels
    # ----------------------------------------

    print("\nGround Truth Class Distribution:")

    print(
        pd.Series(true_labels)
        .value_counts()
        .sort_index()
        .rename({
            0: "Benign",
            1: "Malignant"
        })
    )

    print("\nMST Cluster Distribution:")

    print(
        pd.Series(cluster_labels)
        .value_counts()
        .sort_index()
    )

    # ----------------------------------------
    # 4. Check dimensions
    # ----------------------------------------

    print("\nNumber of samples:")
    print("Feature samples :", X_scaled.shape[0])
    print("Ground truth    :", len(true_labels))
    print("MST labels      :", len(cluster_labels))

    # Make sure dimensions match
    if (
        len(true_labels) != X_scaled.shape[0]
        or len(cluster_labels) != X_scaled.shape[0]
    ):
        raise ValueError(
            "Number of samples and labels do not match."
        )

    # ----------------------------------------
    # 5. Calculate Silhouette Score
    # ----------------------------------------

    silhouette = silhouette_score(
        X_scaled,
        cluster_labels,
        metric="euclidean"
    )

    # ----------------------------------------
    # 6. Calculate Adjusted Rand Index
    # ----------------------------------------

    ari = adjusted_rand_score(
        true_labels,
        cluster_labels
    )

    # ----------------------------------------
    # 7. Calculate Normalized Mutual Information
    # ----------------------------------------

    nmi = normalized_mutual_info_score(
        true_labels,
        cluster_labels
    )

    # ----------------------------------------
    # 8. Display results
    # ----------------------------------------

    print("\n============================================")
    print("       MST CLUSTERING PERFORMANCE")
    print("============================================")

    print(
        f"Silhouette Score              : {silhouette:.4f}"
    )

    print(
        f"Adjusted Rand Index (ARI)     : {ari:.4f}"
    )

    print(
        f"Normalized Mutual Information : {nmi:.4f}"
    )


ERROR: Diagnosis column not found.

Available columns:
['Unnamed: 0', 'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.area_mean', 'x.smoothness_mean', 'x.compactness_mean', 'x.concavity_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.radius_se', 'x.texture_se', 'x.perimeter_se', 'x.area_se', 'x.smoothness_se', 'x.compactness_se', 'x.concavity_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.texture_worst', 'x.perimeter_worst', 'x.area_worst', 'x.smoothness_worst', 'x.compactness_worst', 'x.concavity_worst', 'x.concave_pts_worst', 'x.symmetry_worst', 'x.fractal_dim_worst', 'y']


# Task 8: Visualize MST and Graph-Based Clusters

* Select two representative features (e.g., mean radius vs. mean texture) for 2D plotting.
 * Plot 1: Unclustered standardized data points.
* Plot 2: Minimum Spanning Tree (MST) edge connections overlaid on the data.
* Plot 3: Final clusters formed after cutting the $K - 1$ largest edge(s).

In [ ]:
# ============================================
# Task 8: Visualize MST and Graph-Based Clusters
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------
# 1. Find radius_mean and texture_mean safely
# --------------------------------------------

# Clean column names
df_mst.columns = df_mst.columns.str.strip()

# Create case-insensitive column lookup
column_lookup = {
    col.lower().strip(): col
    for col in df_mst.columns
}

radius_col = column_lookup.get("radius_mean")
texture_col = column_lookup.get("texture_mean")

if radius_col is None or texture_col is None:
    print("ERROR: radius_mean or texture_mean was not found.")
    print("\nAvailable columns:")
    print(df_mst.columns.tolist())

else:

    print("Radius column :", radius_col)
    print("Texture column:", texture_col)

    # ----------------------------------------
    # 2. Get standardized feature values
    # ----------------------------------------

    # X_scaled is a DataFrame created in Task 3
    radius_scaled = X_scaled[radius_col].to_numpy()
    texture_scaled = X_scaled[texture_col].to_numpy()

    # ----------------------------------------
    # 3. Create 3 subplots
    # ----------------------------------------

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(21, 6)
    )

    # ========================================
    # PLOT 1: Unclustered Standardized Data
    # ========================================

    axes[0].scatter(
        radius_scaled,
        texture_scaled,
        color="steelblue",
        s=40,
        alpha=0.75,
        edgecolors="black",
        linewidths=0.3
    )

    axes[0].set_title(
        "Plot 1: Standardized Data",
        fontsize=14,
        fontweight="bold"
    )

    axes[0].set_xlabel(
        "Mean Radius (Standardized)"
    )

    axes[0].set_ylabel(
        "Mean Texture (Standardized)"
    )

    axes[0].grid(
        alpha=0.3
    )

    # ========================================
    # PLOT 2: MST Edge Connections
    # ========================================

    # Plot data points first
    axes[1].scatter(
        radius_scaled,
        texture_scaled,
        color="lightgray",
        s=35,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.3,
        zorder=2
    )

    # Draw MST edges
    for node1, node2, weight in mst_edges:

        axes[1].plot(
            [
                radius_scaled[node1],
                radius_scaled[node2]
            ],
            [
                texture_scaled[node1],
                texture_scaled[node2]
            ],
            color="red",
            linewidth=0.7,
            alpha=0.45,
            zorder=1
        )

    axes[1].set_title(
        "Plot 2: Minimum Spanning Tree",
        fontsize=14,
        fontweight="bold"
    )

    axes[1].set_xlabel(
        "Mean Radius (Standardized)"
    )

    axes[1].set_ylabel(
        "Mean Texture (Standardized)"
    )

    axes[1].grid(
        alpha=0.3
    )

    # ========================================
    # PLOT 3: Final MST Clusters
    # ========================================

    scatter = axes[2].scatter(
        radius_scaled,
        texture_scaled,
        c=cluster_labels,
        cmap="coolwarm",
        s=45,
        alpha=0.85,
        edgecolors="black",
        linewidths=0.3
    )

    axes[2].set_title(
        "Plot 3: MST-Based Clusters (K=2)",
        fontsize=14,
        fontweight="bold"
    )

    axes[2].set_xlabel(
        "Mean Radius (Standardized)"
    )

    axes[2].set_ylabel(
        "Mean Texture (Standardized)"
    )

    axes[2].grid(
        alpha=0.3
    )

    # Colorbar
    colorbar = plt.colorbar(
        scatter,
        ax=axes[2]
    )

    colorbar.set_label(
        "MST Cluster"
    )

    # ----------------------------------------
    # 4. Display plots
    # ----------------------------------------

    plt.tight_layout()
    plt.show()


ERROR: radius_mean or texture_mean was not found.

Available columns:
['Unnamed: 0', 'x.radius_mean', 'x.texture_mean', 'x.perimeter_mean', 'x.area_mean', 'x.smoothness_mean', 'x.compactness_mean', 'x.concavity_mean', 'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean', 'x.radius_se', 'x.texture_se', 'x.perimeter_se', 'x.area_se', 'x.smoothness_se', 'x.compactness_se', 'x.concavity_se', 'x.concave_pts_se', 'x.symmetry_se', 'x.fractal_dim_se', 'x.radius_worst', 'x.texture_worst', 'x.perimeter_worst', 'x.area_worst', 'x.smoothness_worst', 'x.compactness_worst', 'x.concavity_worst', 'x.concave_pts_worst', 'x.symmetry_worst', 'x.fractal_dim_worst', 'y']


# Conclusion

The experiment successfully demonstrated the implementation of Graph-Based Clustering using a Minimum Spanning Tree (MST) on the Breast Cancer Wisconsin (Diagnostic) dataset. The dataset was preprocessed, cleaned, and standardized before computing pairwise distances. The MST was generated, and by removing the largest edge(s), clusters representing tumor categories were obtained, evaluated using clustering metrics, and visualized.